# Day 8 · Exercise 1: Build a Label Set

**What you'll build:** `build_classification_prompt(text: str, labels: list[str]) -> str` — a function that assembles a complete classification prompt embedding the label list and an output constraint so an LLM returns exactly one label.

**Why it matters:** Every zero-shot classifier you build this week starts with a well-formed prompt; getting this function right means every downstream classifier inherits a consistent, reliable structure.

## Your Implementation

In [ ]:
def build_classification_prompt(text: str, labels: list[str]) -> str:
    """Build a classification prompt that instructs an LLM to assign one label.

    The returned string contains the full text the caller should send as the
    user message in an ollama.chat() call.  It embeds (1) the fixed label set
    so the model knows its options, (2) a hard output constraint so the model
    replies with exactly one label and nothing else, and (3) the text to
    classify.

    Args:
        text:   The input text to be classified.
        labels: A non-empty list of label strings (e.g. ['positive', 'negative',
                'neutral']).  Each label must be a non-empty string.

    Returns:
        A single string prompt ready to send to an LLM as the user message.
        The prompt contains all labels joined by commas, an instruction to reply
        with exactly one label and no other text, and the input text.

    Example:
        >>> build_classification_prompt("I love this!", ["positive", "negative"])
        'Classify the following text.\nLabels: positive, negative\nReply with exactly one label from the list above. Output nothing else.\n\nText: I love this!'
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(build_classification_prompt), 'build_classification_prompt is not defined or not callable'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: return type is str
    try:
        result = build_classification_prompt('Great product!', ['positive', 'negative', 'neutral'])
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        print(f'{_PASS} Check 2/{total}: returns a string')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return

    # Check 3: all labels appear in the prompt
    try:
        labels = ['billing', 'technical-issue', 'feature-request']
        prompt = build_classification_prompt('My invoice is wrong.', labels)
        missing = [lbl for lbl in labels if lbl not in prompt]
        assert not missing, f'labels missing from prompt: {missing}'
        print(f'{_PASS} Check 3/{total}: all labels appear in the returned prompt')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: the input text appears in the prompt
    try:
        text = 'The app keeps crashing on startup.'
        prompt = build_classification_prompt(text, ['bug', 'feedback'])
        assert text in prompt, 'input text not found in the returned prompt'
        print(f'{_PASS} Check 4/{total}: input text is embedded in the prompt')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Right now the function builds a prompt string that lives in the user message.  Many production classifiers put the label list and output constraint in the **system** prompt instead, keeping the user message as just the raw text to classify.

Refactor `build_classification_prompt` (or write a companion `build_system_prompt(labels: list[str]) -> str`) so the label instruction is in the system role.  On Day 9 you will see why this separation matters when you add few-shot examples to the user turn.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def build_classification_prompt(text: str, labels: list[str]) -> str:
    label_str = ', '.join(labels)
    return (
        f'Classify the following text.\n'
        f'Labels: {label_str}\n'
        f'Reply with exactly one label from the list above. Output nothing else.\n'
        f'\n'
        f'Text: {text}'
    )
```

**Why this works:** Joining the labels into a comma-separated string gives the model a compact, unambiguous list of its only valid outputs.  The explicit output constraint — "Reply with exactly one label … Output nothing else" — applies the technique from Day 6: you are not asking the model to generate prose, you are forcing a single structured decision.  Embedding the input text last keeps it easy to scan and mirrors the pattern the lesson's `classify_sentiment` example uses, so the two pieces of code read consistently alongside each other.
</details>